# Pipeline evaluation — public fixture path

This public adaptation applies the extraction, retrieval, and answer-checking lessons from my Pfizer externship to the fictional documents in `data/public-fixtures/`. It loads the matching public answer key; the historical evaluation questions and metrics are not reproduced.

Run from the candidate root, or from `notebooks/`; the code below resolves either location. The configured embedding and Mistral weights download when the model setup cell runs. This GPU-oriented model path was not run in the local offline check. For a small deterministic check without model downloads, run `python scripts/offline_demo.py` from the candidate root (`python ../scripts/offline_demo.py` from `notebooks/`). That lexical check is not an embedding or LLM evaluation.


In [ ]:
!pip install -q llama-index llama-index-embeddings-huggingface llama-index-llms-llama-cpp
!pip install -q llama-cpp-python PyPDF2 langchain-text-splitters
!pip install -q pytesseract pdf2image
!apt-get -qq install -y tesseract-ocr poppler-utils

In [ ]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.llama_cpp import LlamaCPP

# Embedding model — used in this adapted evaluation
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# Open-source LLM — Mistral-7B-Instruct v0.2, Q4_K_M GGUF, with GPU offload configured
Settings.llm = LlamaCPP(
    model_url="https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf",
    temperature=0.1,
    max_new_tokens=256,
    context_window=3900,
    model_kwargs={"n_gpu_layers": -1},   # -1 = offload all layers to the T4
    verbose=False,
)
print("Models loaded.")

In [ ]:
from PyPDF2 import PdfReader
from pdf2image import convert_from_path
import pytesseract
from langchain_text_splitters import RecursiveCharacterTextSplitter
from llama_index.core import Document, VectorStoreIndex

from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "data/public-fixtures").exists():
    ROOT = ROOT.parent
PDF_PATH = str(ROOT / "data/public-fixtures/synthetic-quality-documents.pdf")

def extract_page_text(reader, page_num, pdf_path):
    """Digital text first; fall back to OCR if the page has no text layer."""
    text = reader.pages[page_num].extract_text() or ""
    if len(text.strip()) < 20:  # near-empty → scanned page, OCR it
        images = convert_from_path(pdf_path, first_page=page_num + 1, last_page=page_num + 1)
        text = pytesseract.image_to_string(images[0])
    return text

reader = PdfReader(PDF_PATH)
splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100)

documents = []
for page_num in range(len(reader.pages)):
    page_text = extract_page_text(reader, page_num, PDF_PATH)
    chunks = splitter.split_text(page_text)
    for i, chunk in enumerate(chunks):
        documents.append(Document(
            text=chunk,
            metadata={
                "doc_type": "synthetic_quality_fixture",
                "chunk_index": i,
                "page_start": page_num,
                "page_end": page_num,
                "source_file": "synthetic-quality-documents.pdf",
            },
        ))

index = VectorStoreIndex.from_documents(documents)
print(f"Indexed {len(reader.pages)} pages into {len(documents)} chunks.")

In [ ]:
import json
from pathlib import Path
_fixture_key = json.loads((ROOT / "data/public-fixtures/answer-key.json").read_text())
GROUND_TRUTH = [
    {"question": case["question"],
     "expected_keywords": case["expected_answer"].get("contains", []),
     "expected_pages": [case["expected_page"] - 1] if case["expected_page"] is not None else []}
    for case in _fixture_key["questions"]
    if case["expected_answer"].get("status") == "answered"
]


In [ ]:
import time

retriever = index.as_retriever(similarity_top_k=4)
results = []

for item in GROUND_TRUTH:
    q = item["question"]
    t0 = time.time()

    nodes = retriever.retrieve(q)
    retrieved_pages = set()
    for n in nodes:
        ps = n.node.metadata.get("page_start")
        pe = n.node.metadata.get("page_end", ps)
        if ps is not None:
            retrieved_pages.update(range(ps, pe + 1))
    recall_hit = any(p in retrieved_pages for p in item["expected_pages"])

    context = "\n\n".join(n.node.get_content() for n in nodes)
    prompt = (f"Use ONLY the context to answer. Cite sources.\n\n"
              f"Context:\n{context}\n\nQuestion: {q}\nAnswer:")
    answer = Settings.llm.complete(prompt).text.strip()
    latency = time.time() - t0

    em = all(kw.lower() in answer.lower() for kw in item["expected_keywords"])
    results.append({"em": em, "recall": recall_hit, "latency": latency})
    print(f"Q: {q}\n  EM: {em} | Recall@4: {recall_hit} | {latency:.1f}s\n  A: {answer[:150]}...\n")

n = len(results)
print("=" * 45)
print(f"{'Metric':<24}{'Public fixture (model run)':>18}")
print("-" * 45)
print(f"{'Exact Match %':<24}{100*sum(r['em'] for r in results)/n:>16.0f}%")
print(f"{'Retrieval Recall@4':<24}{100*sum(r['recall'] for r in results)/n:>16.0f}%")
print(f"{'Avg Latency (sec)':<24}{sum(r['latency'] for r in results)/n:>17.0f}")
print("=" * 45)

In [ ]:
import time

retriever = index.as_retriever(similarity_top_k=4)
K = 4
recall_hits = hit_hits = 0
precision_sum = rr_sum = 0.0
retrieval_ms = []

for item in GROUND_TRUTH:
    exp = set(item["expected_pages"])
    t0 = time.time()
    nodes = retriever.retrieve(item["question"])
    retrieval_ms.append((time.time() - t0) * 1000)

    ranks_rel, relevant = [], 0
    for rank, nd in enumerate(nodes, start=1):
        ps = nd.node.metadata.get("page_start")
        pe = nd.node.metadata.get("page_end", ps)
        pages = set(range(ps, pe + 1)) if ps is not None else set()
        if pages & exp:
            relevant += 1
            ranks_rel.append(rank)

    if relevant > 0:
        recall_hits += 1
        hit_hits += 1
    precision_sum += relevant / len(nodes)
    rr_sum += (1 / ranks_rel[0]) if ranks_rel else 0

n = len(GROUND_TRUTH)
print(f"Recall@{K}:              {100*recall_hits/n:.1f}%")
print(f"Hit Rate:              {100*hit_hits/n:.1f}%")
print(f"Precision@{K}:           {100*precision_sum/n:.1f}%")
print(f"MRR:                   {rr_sum/n:.2f}")
print(f"Avg Retrieval Latency: {sum(retrieval_ms)/n:.0f} ms")


In [ ]:
q = "What is the expiration date on certificate CERT-FIC-202?"
nodes = retriever.retrieve(q)
print(f"QUERY: {q}\n")
for i, nd in enumerate(nodes, 1):
    pg = nd.node.metadata.get("page_start")
    print(f"Chunk {i}  (Similarity: {nd.score:.2f})  [page {pg}]")
    print("  " + nd.node.get_content()[:180].replace("\n", " ").strip() + "…")
    print()
